In [1]:
import sqlite3
import pandas as pd

In [ ]:
cursor.execute("DROP TABLE IF EXISTS Articles")
cursor.execute("DROP TABLE IF EXISTS Labels")

In [3]:
conn = sqlite3.connect("news.db")
cursor = conn.cursor()
cursor.executescript("""
    CREATE TABLE IF NOT EXISTS Articles(
        article_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title VARCHAR(100),
        text TEXT,
        subject VARCHAR(20),
        date DATE
    );
    CREATE TABLE IF NOT EXISTS Labels(
        article_id INTEGER PRIMARY KEY,
        label VARCHAR(5) NOT NULL CHECK(label IN ('fake', 'real')),
        FOREIGN KEY (article_id) REFERENCES Articles(article_id)
    );
""")
conn.commit()

tables created successfully


In [4]:
fake_df = pd.read_csv("data/Fake.csv")
real_df = pd.read_csv("data/True.csv")

fake_df["label"] = "fake"
real_df["label"] = "real"

df = pd.concat([fake_df, real_df], ignore_index=True)

print(f"Total articles: {len(df)}")
print(f"  Fake: {len(fake_df)}")
print(f"  Real: {len(real_df)}")
df.head()

Total articles: 44898
  Fake: 23481
  Real: 21417


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",fake
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",fake
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",fake
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",fake
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",fake


In [5]:
articles_df = df[["title", "text", "subject", "date"]]
articles_df.to_sql("Articles", conn, if_exists="append", index=False)

cursor.execute("SELECT article_id FROM Articles ORDER BY article_id")
ids = [row[0] for row in cursor.fetchall()] # extract each article_id from list of tuples

labels_df = pd.DataFrame({"article_id": ids, "label": df["label"].values})
labels_df.to_sql("Labels", conn, if_exists="append", index=False)

conn.commit()
print(f"Ingested {len(df)} articles into the database.")

Ingested 44898 articles into the database.


In [6]:
# making sure everything worked right
result = pd.read_sql_query("""
    SELECT l.label, COUNT(*) AS article_count
    FROM Articles a
    JOIN Labels l ON a.article_id = l.article_id
    GROUP BY l.label
""", conn)

result

,label,article_count
0,fake,23481
1,real,21417


In [7]:
conn.close()

In [2]:
fake_df = pd.read_csv("data/Fake.csv")
real_df = pd.read_csv("data/True.csv")

fake_df.head(50).to_csv("data/Fake_sample.csv", index=False)
real_df.head(50).to_csv("data/True_sample.csv", index=False)

print("Sample files created.")

Sample files created.
